<a href="https://colab.research.google.com/github/sruthi-kurra/dpo-vs-sft-qwen/blob/main/notebooks/02_Supervised_Fine_Tuning_QLoRA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Cell 1 — Setup

In [1]:
!pip install -q transformers accelerate peft bitsandbytes trl datasets

import json
import random
import numpy as np
import torch
from pathlib import Path

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
DATA_DIR = Path("data")
OUTPUT_DIR = Path("checkpoints/sft")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 842.4/842.4 kB 22.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 12.9 MB/s eta 0:00:00
CUDA available: True
GPU: Tesla T4


## Cell 2 — Load UltraFeedback Dataset & Create Deterministic Train/Eval Split

In [2]:
from datasets import load_dataset

print("Loading UltraFeedback (train_sft)...")

dataset = load_dataset(
    "HuggingFaceH4/ultrafeedback_binarized",
    split="train_sft"
)

# Deterministic split (same every run)
dataset = dataset.shuffle(seed=SEED)

main = dataset.select(range(2500))

train_ds = main.select(range(2250))
eval_ds = main.select(range(2250, 2500))

print(f"Train examples: {len(train_ds)}")
print(f"Eval examples : {len(eval_ds)}")

Loading UltraFeedback (train_sft)...


README.md:   0%|          | 0.00/6.53k [00:00<?, ?B/s]

data/train_prefs-00000-of-00001.parquet:   0%|          | 0.00/226M [00:00<?, ?B/s]

data/test_prefs-00000-of-00001.parquet:   0%|          | 0.00/7.29M [00:00<?, ?B/s]

data/test_sft-00000-of-00001.parquet:   0%|          | 0.00/3.72M [00:00<?, ?B/s]

...

Generating test_prefs split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Generating test_sft split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Generating train_gen split:   0%|          | 0/61135 [00:00<?, ? examples/s]

Generating test_gen split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Train examples: 2250
Eval examples : 250


## Cell 3 — Apply Qwen Chat Template & Prepare Training Dataset

In [6]:
from transformers import AutoTokenizer

# Load tokenizer here directly so this cell never depends on Cell 4
# having been run first (survives runtime restarts / out-of-order execution)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def format_example(example):
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False
    )
    return {"text": text}

train_ds = train_ds.map(format_example)
eval_ds = eval_ds.map(format_example)

print(f"Tokenizer loaded: {MODEL_NAME}")
print(train_ds[0]["text"][:500])

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Map:   0%|          | 0/2250 [00:00<?, ? examples/s]

Map:   0%|          | 0/250 [00:00<?, ? examples/s]

Tokenizer loaded: Qwen/Qwen2.5-0.5B-Instruct
<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
Do you know something about crystallography and structure factor?<|im_end|>
<|im_start|>assistant
Crystallography is the science of the arrangement of atoms in solids. It is a vast and interdisciplinary field that has applications in physics, chemistry, materials science, biology, and engineering.

The structure factor is a mathematical function that is used to describe the diffract


## Cell 4 — Load Qwen2.5-0.5B-Instruct with 4-bit QLoRA

In [7]:
from transformers import (
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
)

device = "cuda" if torch.cuda.is_available() else "cpu"

# (tokenizer already loaded in Cell 3 — no need to reload here)

# ------------------------------------------------------------
# 4-bit quantization config
# ------------------------------------------------------------

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

# ------------------------------------------------------------
# Load base model
# ------------------------------------------------------------

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)

model.gradient_checkpointing_enable()
model = prepare_model_for_kbit_training(model)

# ------------------------------------------------------------
# LoRA configuration
# ------------------------------------------------------------

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
)

model = get_peft_model(model, lora_config)

# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

model.print_trainable_parameters()

print("\nModel loaded successfully.")
print(f"Model: {MODEL_NAME}")
print(f"Device: {device}")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

trainable params: 8,798,208 || all params: 502,830,976 || trainable%: 1.7497

Model loaded successfully.
Model: Qwen/Qwen2.5-0.5B-Instruct
Device: cuda


## Cell 5 — Prepare Dataset for SFTTrainer

In [8]:
train_ds = train_ds.remove_columns(
    [c for c in train_ds.column_names if c != "text"]
)

eval_ds = eval_ds.remove_columns(
    [c for c in eval_ds.column_names if c != "text"]
)

print(train_ds.column_names)
print(eval_ds.column_names)

['text']
['text']


## Cell 6 — Configure SFT Training

In [9]:
from trl import SFTConfig

total_train_examples = len(train_ds)
effective_batch_size = 4 * 4  # per_device_train_batch_size * gradient_accumulation_steps
steps_per_epoch = total_train_examples // effective_batch_size
warmup_steps = max(1, int(0.03 * steps_per_epoch))  # same 3% warmup, just pre-computed

sft_config = SFTConfig(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=1,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=warmup_steps,   # replaces warmup_ratio
    weight_decay=0.01,
    max_grad_norm=0.3,

    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=50,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    bf16=torch.cuda.is_available(),
    optim="paged_adamw_8bit",
    report_to="none",
    seed=SEED,

    max_length=1024,
    dataset_text_field="text",
    packing=False,
)

print("SFT training config ready.")
print(f"Steps per epoch: {steps_per_epoch}, warmup_steps: {warmup_steps}")
print(f"Effective batch size: {sft_config.per_device_train_batch_size * sft_config.gradient_accumulation_steps}")
print(f"Output dir: {sft_config.output_dir}")

SFT training config ready.
Steps per epoch: 140, warmup_steps: 4
Effective batch size: 16
Output dir: checkpoints/sft


## Cell 7 — Train SFT Model

In [10]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    processing_class=tokenizer,   # renamed from `tokenizer=` in trl>=0.16
)

print("Starting SFT training...")
train_result = trainer.train()

print("\nTraining complete.")
print(f"Final train loss: {train_result.training_loss:.4f}")

Adding EOS to train dataset:   0%|          | 0/2250 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/2250 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/2250 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/250 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/250 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/250 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Starting SFT training...


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
50,1.551651,1.594175,1.581622,363748.000000,0.634615
100,1.550667,1.577967,1.566332,728663.000000,0.637218
141,1.562880,1.575069,1.570220,1016126.000000,0.638172



Training complete.
Final train loss: 1.6011


## Cell 8 — Evaluate & Save SFT Model

In [11]:
# ------------------------------------------------------------
# Cell 8 — Evaluate & Save SFT Model
# ------------------------------------------------------------

# Final eval on held-out set
eval_metrics = trainer.evaluate()
print("Final eval metrics:")
for k, v in eval_metrics.items():
    print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

# ------------------------------------------------------------
# Save LoRA adapter + tokenizer
# ------------------------------------------------------------

SFT_ADAPTER_DIR = OUTPUT_DIR / "sft_adapter_final"
SFT_ADAPTER_DIR.mkdir(parents=True, exist_ok=True)

trainer.save_model(str(SFT_ADAPTER_DIR))
tokenizer.save_pretrained(str(SFT_ADAPTER_DIR))

print(f"\nSFT LoRA adapter saved to: {SFT_ADAPTER_DIR}")
print("Contents:", [p.name for p in SFT_ADAPTER_DIR.iterdir()])

# ------------------------------------------------------------
# Quick sanity check: generate a sample response
# ------------------------------------------------------------

model.eval()
sample_prompt = eval_ds[0]["text"].split("<|im_start|>assistant")[0] + "<|im_start|>assistant\n"
inputs = tokenizer(sample_prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    output = model.generate(
        **inputs,
        max_new_tokens=150,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
    )

generated = tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print("\nSample generation from fine-tuned SFT model:")
print(generated)

Training Loss,Validation Loss,Step,Entropy,Num Tokens,Mean Token Accuracy
1.562880,1.575069,141,1.570220,1016126.000000,0.638172


Final eval metrics:
  eval_loss: 1.5751
  eval_entropy: 1.5702
  eval_num_tokens: 1016126.0000
  eval_mean_token_accuracy: 0.6382

SFT LoRA adapter saved to: checkpoints/sft/sft_adapter_final
Contents: ['chat_template.jinja', 'adapter_config.json', 'training_args.bin', 'adapter_model.safetensors', 'tokenizer_config.json', 'README.md', 'tokenizer.json']


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)



Sample generation from fine-tuned SFT model:
168


## Cell 9 — Free Memory & Load DPO Preference Dataset

In [18]:
# ------------------------------------------------------------
# Cell 9 — Free SFT Memory & Prepare DPO Preference Dataset
# ------------------------------------------------------------

import gc

# Free the SFT training model/trainer from GPU memory before DPO
del trainer
del model
gc.collect()
torch.cuda.empty_cache()

print(f"GPU memory after cleanup: {torch.cuda.memory_allocated() / 1e9:.2f} GB allocated")

# ------------------------------------------------------------
# Load UltraFeedback with chosen/rejected pairs for DPO
# ------------------------------------------------------------

from datasets import load_dataset

print("Loading UltraFeedback (train_prefs / test_prefs)...")

dpo_dataset = load_dataset(
    "HuggingFaceH4/ultrafeedback_binarized",
    split="train_prefs"
)

# Same deterministic split pattern as Cell 2
dpo_dataset = dpo_dataset.shuffle(seed=SEED)

dpo_main = dpo_dataset.select(range(2500))
dpo_train_ds = dpo_main.select(range(2250))
dpo_eval_ds = dpo_main.select(range(2250, 2500))

print(f"DPO train examples: {len(dpo_train_ds)}")
print(f"DPO eval examples: {len(dpo_eval_ds)}")
print(f"\nColumns available: {dpo_dataset.column_names}")
print(f"\nSample chosen: {dpo_train_ds[0]['chosen']}")
print(f"\nSample rejected: {dpo_train_ds[0]['rejected']}")

GPU memory after cleanup: 0.56 GB allocated
Loading UltraFeedback (train_prefs / test_prefs)...
DPO train examples: 2250
DPO eval examples: 250

Columns available: ['prompt', 'prompt_id', 'chosen', 'rejected', 'messages', 'score_chosen', 'score_rejected']

Sample chosen: [{'content': 'Do you know something about crystallography and structure factor?', 'role': 'user'}, {'content': 'Crystallography is the science of the arrangement of atoms in solids. It is a vast and interdisciplinary field that has applications in physics, chemistry, materials science, biology, and engineering.\n\nThe structure factor is a mathematical function that is used to describe the diffraction of waves by a crystal. It is a complex number that is related to the atomic positions in the crystal.\n\nThe structure factor can be used to calculate the intensity of the diffracted waves. This information can be used to determine the atomic positions in the crystal and to study the structure of materials.\n\nCrystallogr